# 📱 Task 3 - Size vs Rating Bubble Chart Analysis

### Goal:
Compare the relationship between app size (in MB) and average rating, using bubble size for the number of installs.

### Rules:
1. Rating > 3.5
2. Reviews > 500
3. Installs > 50,000
4. Exclude apps containing the letter "S" (case-insensitive)
5. Sentiment subjectivity > 0.5 (Simulated)
6. Categories: Game, Beauty, Business, Comics, Communication, Dating, Entertainment, Social, Events
7. Translate Category names:
   * Beauty -> Hindi (सुंदरता)
   * Business -> Tamil (வணிகம்)
   * Dating -> German (Partnersuche)
8. Highlight "GAME" category in **pink**.
9. Only display between **5 PM and 7 PM IST** (Time bypass available for testing).

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import random
from datetime import datetime
import pytz
import plotly.io as pio

# Set default renderer to browser
pio.renderers.default = "browser"

In [ ]:
# Load dataset
dataset = pd.read_csv("../Play Store Data.csv")
dataset.info()

In [ ]:
# Convert Size to MB
dataset["Size"] = dataset["Size"].astype(str)

def size_to_mb(x):
    if "M" in x:
        return float(x.replace("M", ""))
    elif "k" in x:
        return float(x.replace("k", "")) / 1024
    else:
        return np.nan

dataset["Size_MB"] = dataset["Size"].apply(size_to_mb)
print(dataset["Size_MB"].head())

In [ ]:
# Clean Installs, Reviews, and Ratings
dataset["Installs"] = dataset["Installs"].str.replace("[+,]", "", regex=True)
dataset["Installs"] = pd.to_numeric(dataset["Installs"], errors="coerce")
dataset["Reviews"] = pd.to_numeric(dataset["Reviews"], errors="coerce")
dataset["Rating"] = pd.to_numeric(dataset["Rating"], errors="coerce")

# Drop rows with missing values in required fields
dataset = dataset.dropna(subset=["Rating", "Installs", "Reviews", "Size_MB"])

In [ ]:
# Simulate Sentiment Subjectivity (since it is not present in the Play Store Data CSV)
np.random.seed(42)
dataset["Sentiment_Subjectivity"] = np.random.uniform(0.0, 1.0, size=len(dataset))

In [ ]:
# Apply all required filters
target_categories = ["GAME", "BEAUTY", "BUSINESS", "COMICS", "COMMUNICATION", "DATING", "ENTERTAINMENT", "SOCIAL", "EVENTS"]

filtered_df = dataset[
    (dataset["Rating"] > 3.5) &
    (dataset["Reviews"] > 500) &
    (dataset["Installs"] > 50000) &
    (~dataset["App"].str.contains("S", case=False, na=False)) &
    (dataset["Sentiment_Subjectivity"] > 0.5) &
    (dataset["Category"].isin(target_categories))
].copy()

print(f"Filtered dataset count: {len(filtered_df)} apps")

In [ ]:
# Apply Translations
translation_map = {
    "BEAUTY": "सुंदरता (Beauty)",
    "BUSINESS": "வணிகம் (Business)",
    "DATING": "Dating (German)"
}

filtered_df["Category"] = filtered_df["Category"].replace(translation_map)

In [ ]:
# Check time logic (5 PM to 7 PM IST) and plot graph
now = datetime.now(pytz.timezone('Asia/Kolkata'))

# Change bypass_time_check to False to strictly enforce the 5 PM - 7 PM IST window
bypass_time_check = True

if bypass_time_check or (17 <= now.hour < 19):
    fig = px.scatter(
        filtered_df,
        x="Rating",
        y="Size_MB",
        size="Installs",
        color="Category",
        hover_name="App",
        title="App Size vs Rating (Bubble Size = Installs, Game category highlighted in pink)",
        color_discrete_map={"GAME": "pink"},
        size_max=50
    )
    fig.update_layout(width=1000, height=600)
    fig.show()
else:
    print("⏱️ This chart is only available between 5 PM and 7 PM IST.")